# FAANG + MSFT + S&P 500 vs BTC Close Price Correlation (Past 5 Years)

This notebook visualizes the close prices of FAANG + MSFT + S&P 500 (META, AAPL, AMZN, NFLX, GOOGL, MSFT, ^GSPC) and Bitcoin (BTC-USD) over the past 5 years to explore their correlation.

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## 2. Load Data

In [2]:
end_date = datetime.now()
start_date = end_date - timedelta(days=5*365)

# FAANG + MSFT + S&P 500: META, AAPL, AMZN, NFLX, GOOGL, MSFT, ^GSPC + BTC
tickers = ['META', 'AAPL', 'AMZN', 'NFLX', 'GOOGL', 'MSFT', '^GSPC', 'BTC-USD']
data = yf.download(tickers, start=start_date, end=end_date, progress=False, auto_adjust=True)

# Handle multi-column structure from yfinance
if isinstance(data.columns, pd.MultiIndex):
    close = data['Close'].copy()
else:
    close = data[['Close']].copy() if 'Close' in data.columns else data.copy()

close = close.dropna(how='all')
print(f"Loaded {len(close)} trading days from {close.index.min().date()} to {close.index.max().date()}")
close.tail()

Loaded 1825 trading days from 2021-03-02 to 2026-02-28


Ticker,AAPL,AMZN,BTC-USD,GOOGL,META,MSFT,NFLX,^GSPC
Date,,,,,,,,
2026-02-24,272.140015,208.559998,64080.042969,310.899994,639.299988,389.000000,78.040001,6890.069824
2026-02-25,274.230011,210.639999,67960.125000,312.899994,653.690002,400.600006,82.699997,6946.129883
2026-02-26,272.950012,207.919998,67453.773438,307.380005,657.010010,401.720001,84.589996,6908.859863
2026-02-27,264.179993,210.000000,65881.796875,311.760010,648.179993,392.739990,96.239998,6878.879883
2026-02-28,NaN,NaN,64729.484375,NaN,NaN,NaN,NaN,NaN


## 3. Normalized Close Prices (All FAANG + BTC)

In [3]:
# Normalize to 100 at start. FAANG on left axis, BTC on right (different scales). Click legend to toggle.
normalized = (close / close.iloc[0]) * 100

colors = {'META': '#0668E1', 'AAPL': '#555555', 'AMZN': '#FF9900', 'NFLX': '#E50914', 'GOOGL': '#4285F4', 'MSFT': '#0078d4', '^GSPC': '#2e7d32', 'GSPC': '#2e7d32', 'BTC-USD': '#f7931a'}
fig = make_subplots(specs=[[{"secondary_y": True}]])
faang_cols = [c for c in close.columns if c != 'BTC-USD']
for col in faang_cols:
    fig.add_trace(go.Scatter(x=normalized.index, y=normalized[col], name=col, line=dict(color=colors.get(col, '#333'), width=2)), secondary_y=False)
fig.add_trace(go.Scatter(x=normalized.index, y=normalized['BTC-USD'], name='BTC-USD', line=dict(color='#f7931a', width=2)), secondary_y=True)
fig.add_hline(y=100, line_dash='dash', line_color='gray', opacity=0.5, secondary_y=False)
fig.add_hline(y=100, line_dash='dash', line_color='gray', opacity=0.5, secondary_y=True)
fig.update_layout(
    title='FAANG + MSFT + S&P 500 (left) vs BTC (right) Normalized Close — Click legend to compare pairs',
    xaxis_title='Date', hovermode='x unified', legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    height=500, template='plotly_white'
)
fig.update_yaxes(title_text='FAANG + MSFT + S&P 500 Indexed (Start=100)', secondary_y=False)
fig.update_yaxes(title_text='BTC Indexed (Start=100)', secondary_y=True)
fig.show()

## 4. Raw Close Prices (BTC vs FAANG Average)

In [4]:
# Raw close prices — FAANG on left axis, BTC on right. Click legend to compare any pair.
fig = make_subplots(specs=[[{"secondary_y": True}]])
faang_cols = [c for c in close.columns if c != 'BTC-USD']
for col in faang_cols:
    fig.add_trace(go.Scatter(x=close.index, y=close[col], name=col, line=dict(width=2)), secondary_y=False)
fig.add_trace(go.Scatter(x=close.index, y=close['BTC-USD'], name='BTC-USD', line=dict(width=2)), secondary_y=True)
fig.update_layout(
    title='FAANG + MSFT + S&P 500 (left) vs BTC (right) Raw Close Prices — Click legend to compare pairs',
    xaxis_title='Date', hovermode='x unified', legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    height=500, template='plotly_white'
)
fig.update_yaxes(title_text='FAANG + MSFT + S&P 500 Close ($)', secondary_y=False)
fig.update_yaxes(title_text='BTC Close ($)', secondary_y=True)
fig.show()

## 5. Rolling Correlation

In [5]:
# Daily returns
returns = close.pct_change(fill_method=None).dropna()

# Rolling 252-day (1 year) correlation of each FAANG with BTC
fig = go.Figure()
colors = {'META': '#0668E1', 'AAPL': '#555555', 'AMZN': '#FF9900', 'NFLX': '#E50914', 'GOOGL': '#4285F4', 'MSFT': '#0078d4', '^GSPC': '#2e7d32', 'GSPC': '#2e7d32'}
for col in [c for c in returns.columns if c != 'BTC-USD']:
    rolling_corr = returns[col].rolling(252).corr(returns['BTC-USD'])
    fig.add_trace(go.Scatter(x=rolling_corr.index, y=rolling_corr, name=f"{col} vs BTC", line=dict(color=colors.get(col, '#333'), width=2)))
fig.add_hline(y=0, line_dash='dash', line_color='gray', opacity=0.5)
fig.update_layout(
    title='Rolling 1-Year Correlation: Each FAANG + MSFT + S&P 500 vs BTC — Click legend to compare pairs',
    xaxis_title='Date', yaxis_title='Correlation',
    hovermode='x unified', legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    height=450, template='plotly_white', yaxis=dict(range=[-1, 1])
)
fig.show()

print("Full-period correlation (daily returns) vs BTC:")
for col in [c for c in returns.columns if c != 'BTC-USD']:
    print(f"  {col}: {returns[col].corr(returns['BTC-USD']):.3f}")

Full-period correlation (daily returns) vs BTC:
  AAPL: 0.305
  AMZN: 0.303
  GOOGL: 0.300
  META: 0.237
  MSFT: 0.358
  NFLX: 0.270
  ^GSPC: 0.420
